In [22]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
# Install required packages (if not already)
!pip install nltk spacy textrank4zh

# Download necessary NLTK data
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# For spaCy model (optional, if using spaCy too)
!python -m spacy download en_core_web_sm


In [23]:
#importing other librariesa
import re

import numpy as np
from nltk import sent_tokenize, word_tokenize

from nltk.cluster.util import cosine_distance

MULTIPLE_WHITESPACE_PATTERN = re.compile(r"\s+", re.UNICODE)

In [24]:
def normalize_whitespace(text):
    """
    Translates multiple whitespace into single space character.
    If there is at least one new line character chunk is replaced
    by single LF (Unix new line) character.
    """
    return MULTIPLE_WHITESPACE_PATTERN.sub(_replace_whitespace, text)


def _replace_whitespace(match):
    text = match.group()

    if "\n" in text or "\r" in text:
        return "\n"
    else:
        return " "


def is_blank(string):
    """
    Returns `True` if string contains only white-space characters
    or is empty. Otherwise `False` is returned.
    """
    return not string or string.isspace()


def get_symmetric_matrix(matrix):
    """
    Get Symmetric matrix
    :param matrix:
    :return: matrix
    """
    return matrix + matrix.T - np.diag(matrix.diagonal())


def core_cosine_similarity(vector1, vector2):
    """
    measure cosine similarity between two vectors
    :param vector1:
    :param vector2:
    :return: 0 < cosine similarity value < 1
    """
    return 1 - cosine_distance(vector1, vector2)


'''
Note: This is not a summarization algorithm. This Algorithm pics top sentences irrespective of the order they appeared.
'''

'\nNote: This is not a summarization algorithm. This Algorithm pics top sentences irrespective of the order they appeared.\n'

In [25]:
class TextRank4Sentences():
    def __init__(self):
        self.damping = 0.85  # damping coefficient, usually is .85
        self.min_diff = 1e-5  # convergence threshold
        self.steps = 100  # iteration steps
        self.text_str = None
        self.sentences = None
        self.pr_vector = None

    def _sentence_similarity(self, sent1, sent2, stopwords=None):
        if stopwords is None:
            stopwords = []

        sent1 = [w.lower() for w in sent1]
        sent2 = [w.lower() for w in sent2]

        all_words = list(set(sent1 + sent2))

        vector1 = [0] * len(all_words)
        vector2 = [0] * len(all_words)

        # build the vector for the first sentence
        for w in sent1:
            if w in stopwords:
                continue
            vector1[all_words.index(w)] += 1

        # build the vector for the second sentence
        for w in sent2:
            if w in stopwords:
                continue
            vector2[all_words.index(w)] += 1

        return core_cosine_similarity(vector1, vector2)

    def _build_similarity_matrix(self, sentences, stopwords=None):
        # create an empty similarity matrix
        sm = np.zeros([len(sentences), len(sentences)])

        for idx1 in range(len(sentences)):
            for idx2 in range(len(sentences)):
                if idx1 == idx2:
                    continue

                sm[idx1][idx2] = self._sentence_similarity(sentences[idx1], sentences[idx2], stopwords=stopwords)

        # Get Symmeric matrix
        sm = get_symmetric_matrix(sm)

        # Normalize matrix by column
        norm = np.sum(sm, axis=0)
        sm_norm = np.divide(sm, norm, where=norm != 0)  # this is ignore the 0 element in norm

        return sm_norm

    def _run_page_rank(self, similarity_matrix):

        pr_vector = np.array([1] * len(similarity_matrix))

        # Iteration
        previous_pr = 0
        for epoch in range(self.steps):
            pr_vector = (1 - self.damping) + self.damping * np.matmul(similarity_matrix, pr_vector)
            if abs(previous_pr - sum(pr_vector)) < self.min_diff:
                break
            else:
                previous_pr = sum(pr_vector)

        return pr_vector

    def _get_sentence(self, index):

        try:
            return self.sentences[index]
        except IndexError:
            return ""

    def get_top_sentences(self, number=5):

        top_sentences = []

        if self.pr_vector is not None:

            sorted_pr = np.argsort(self.pr_vector)
            sorted_pr = list(sorted_pr)
            sorted_pr.reverse()

            index = 0
            for epoch in range(number):
                sent = self.sentences[sorted_pr[index]]
                sent = normalize_whitespace(sent)
                top_sentences.append(sent)
                index += 1

        return top_sentences

    def analyze(self, text, stop_words=None):
        self.text_str = text
        self.sentences = sent_tokenize(self.text_str)

        tokenized_sentences = [word_tokenize(sent) for sent in self.sentences]

        similarity_matrix = self._build_similarity_matrix(tokenized_sentences, stop_words)

        self.pr_vector = self._run_page_rank(similarity_matrix)

In [26]:
import spacy
nlp = spacy.load("en_core_web_sm")


In [3]:
# create spacy
text_str = '''
    Mr. President, I offer you our congratulations on your election as the President of the current session of the General Assembly.
You represent Norway, a country which can take pride in its reputation as peaceful, just and progressive.
Your personal qualifications and your family's dedication to international effort are well known.
I should also like to express our appreciation of the services of your distinguished predecessor, Mrs. Angie Brooks Randolph.
I would also repeat our admiration for U Thant, whose skill and dedication have won him our respect
    '''
import spacy
nlp = spacy.load('en_core_web_sm')
doc = nlp(text_str)

for token in doc:
    print(token.text,'->',token.pos_)


     -> SPACE
Mr. -> PROPN
President -> PROPN
, -> PUNCT
I -> PRON
offer -> VERB
you -> PRON
our -> PRON
congratulations -> NOUN
on -> ADP
your -> PRON
election -> NOUN
as -> ADP
the -> DET
President -> PROPN
of -> ADP
the -> DET
current -> ADJ
session -> NOUN
of -> ADP
the -> DET
General -> PROPN
Assembly -> PROPN
. -> PUNCT

 -> SPACE
You -> PRON
represent -> VERB
Norway -> PROPN
, -> PUNCT
a -> DET
country -> NOUN
which -> PRON
can -> AUX
take -> VERB
pride -> NOUN
in -> ADP
its -> PRON
reputation -> NOUN
as -> ADP
peaceful -> ADJ
, -> PUNCT
just -> ADV
and -> CCONJ
progressive -> ADJ
. -> PUNCT

 -> SPACE
Your -> PRON
personal -> ADJ
qualifications -> NOUN
and -> CCONJ
your -> PRON
family -> NOUN
's -> PART
dedication -> NOUN
to -> ADP
international -> ADJ
effort -> NOUN
are -> AUX
well -> ADV
known -> VERB
. -> PUNCT

 -> SPACE
I -> PRON
should -> AUX
also -> ADV
like -> VERB
to -> PART
express -> VERB
our -> PRON
appreciation -> NOUN
of -> ADP
the -> DET
services -> NOUN
of -> A

In [28]:
from spacy import displacy
displacy.render(doc, style='dep',jupyter=True)


In [4]:
!pip install textrank4zh --quiet


In [5]:
from textrank4zh import TextRank4Sentence
tr4s = TextRank4Sentence()
tr4s.analyze(text_str, lower=True, source='all_filters')
print(tr4s.get_key_sentences(num=5))

Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
DEBUG:jieba:Loading model from cache /tmp/jieba.cache
Loading model cost 4.015 seconds.
DEBUG:jieba:Loading model cost 4.015 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


[{'index': 4, 'sentence': 'I would also repeat our admiration for U Thant, whose skill and dedication have won him our respect', 'weight': 0.2037330954158127}, {'index': 0, 'sentence': 'Mr. President, I offer you our congratulations on your election as the President of the current session of the General Assembly.', 'weight': 0.2}, {'index': 1, 'sentence': 'You represent Norway, a country which can take pride in its reputation as peaceful, just and progressive.', 'weight': 0.2}, {'index': 3, 'sentence': 'I should also like to express our appreciation of the services of your distinguished predecessor, Mrs. Angie Brooks Randolph.', 'weight': 0.2}, {'index': 2, 'sentence': "Your personal qualifications and your family's dedication to international effort are well known.", 'weight': 0.1962669045841871}]
